In [1]:
!nvidia-smi

Fri Aug 21 14:48:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 591.44                 Driver Version: 591.44         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3070 Ti   WDDM  |   00000000:01:00.0  On |                  N/A |
| 43%   42C    P0             87W /  310W |    2097MiB /   8192MiB |      9%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch
from torch import nn

torch.device('cpu'), torch.cuda.device('cuda'), torch.cuda.device('cuda:1')

(device(type='cpu'), device(type='cuda'), device(type='cuda', index=1))

In [3]:
torch.cuda.device_count()

1

In [7]:
torch.cuda.device('cuda'), torch.cuda.device('cuda:1')

(<torch.cuda.device at 0x1875de64af0>, <torch.cuda.device at 0x18711b88d10>)

In [8]:
def try_gpu(i=0):  #@save
    """如果存在，则返回gpu(i)，否则返回cpu()"""
    if torch.cuda.device_count() >= i + 1:
        return torch.device(f'cuda:{i}')
    return torch.device('cpu')

def try_all_gpus():  #@save
    """返回所有可用的GPU，如果没有GPU，则返回[cpu(),]"""
    devices = [torch.device(f'cuda:{i}')
             for i in range(torch.cuda.device_count())]
    return devices if devices else [torch.device('cpu')]

try_gpu(), try_gpu(10), try_all_gpus()

(device(type='cuda', index=0),
 device(type='cpu'),
 [device(type='cuda', index=0)])

In [9]:
x = torch.tensor([1, 2, 3])
x.device

device(type='cpu')

In [13]:
X = torch.ones(2, 3, device=try_gpu())
X.device

device(type='cuda', index=0)

In [15]:
Y = torch.rand(2, 3, device=try_gpu(1))
Y.device

device(type='cpu')

In [17]:
print(X)
print(Y)

tensor([[1., 1., 1.],
        [1., 1., 1.]], device='cuda:0')
tensor([[0.9129, 0.4499, 0.9355],
        [0.8914, 0.4443, 0.1145]])


In [18]:
Z=Y.cuda(0)

In [19]:
print(Z)

tensor([[0.9129, 0.4499, 0.9355],
        [0.8914, 0.4443, 0.1145]], device='cuda:0')


In [21]:
X+Z

tensor([[1.9129, 1.4499, 1.9355],
        [1.8914, 1.4443, 1.1145]], device='cuda:0')

In [22]:
net = nn.Sequential(nn.Linear(3, 1))
net = net.to(device=try_gpu())

In [26]:
print(net(X))

tensor([[-0.6874],
        [-0.6874]], device='cuda:0', grad_fn=<AddmmBackward0>)


In [29]:
net[0].weight.data.device

device(type='cuda', index=0)